# Cross-Validation Strategies in Machine Learning: Hands-on Implementation

## What This Notebook Covers
This notebook is the practical, code-driven counterpart to the **Cross-Validation Strategies README**. We will explore how to evaluate models reliably. You will learn to perform manual K-Fold splits using NumPy indexing, run automated cross-validation using Scikit-learn splitter objects, and analyze how performance fluctuates across folds and hyperparameter settings.

## What You Will Accomplish
- Describe the limits of hold-out validation and explain the threat of validation variance.
- Implement manual K-Fold Cross-Validation from scratch using NumPy index permutation.
- Audit fold distributions to verify why stratification is critical for classification models.
- Build, train, and visualize a comparison of K-Fold, Stratified K-Fold, and Leave-One-Out (LOOCV) cross-validation methods.
- Run empirical sweeps over $K$-fold parameters ($K=3, 5, 10$) to diagnose the bias-variance trade-off.

## Before You Start (Prerequisites)
- Comfort manipulating Python loops and indices.
- Basic awareness of Scikit-learn models (`LogisticRegression`).
- Zero prior cross-validation experience is assumed.

## About the Dataset
We use the benchmark **Iris flower dataset**, containing 150 instances of flowers split evenly across three species: *setosa*, *versicolor*, and *virginica*. For every flower, we have four numeric measurements:
- Sepal length (cm)
- Sepal width (cm)
- Petal length (cm)
- Petal width (cm)

We load it directly using `sklearn.datasets.load_iris`. If you want to explore the dataset outside this notebook, it is also archived on Kaggle:
**https://www.kaggle.com/datasets/uciml/iris**

---

## 1. Setup & Workspace Preparation

### WHY?
Setting up imports at the start of our session avoids path errors and fixes seeds to ensure all stochastic shuffling steps are reproducible.

### HOW?
We import NumPy, Pandas, Matplotlib, Seaborn, and the necessary Scikit-learn validation models, then set seaborn style configurations.

In [ ]:
# Import NumPy for manual fold arithmetic and array operations
import numpy as np

# Import Pandas to display dataframes and stats tables
import pandas as pd

# Import Matplotlib and Seaborn for plotting performance charts
import matplotlib.pyplot as plt
import seaborn as sns

# Import Iris dataset tools and models from sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    LeaveOneOut,
    cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set seaborn style for clean grids
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

# Fix seed for reproducible shuffles
np.random.seed(42)

print("All libraries imported and seed fixed to 42.")

## 2. Dataset Loading & Exploration

### WHY?
Checking class balance and measurement distributions before splitting is essential to verify if standard random splits are safe, or if stratification is required.

### HOW?
We load the Iris dataset, wrap it in a Pandas DataFrame, and print describe statistics.

In [ ]:
# Load iris dataset dictionary from sklearn
iris = load_iris()

# Convert to a pandas DataFrame
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

# Append target codes (0, 1, 2) and target names
df['species'] = iris.target
species_map = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}
df['species_name'] = df['species'].map(species_map)

print("═" * 60)
print(f"Dataset Shape : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Features      : {iris.feature_names}")
print(f"Target Classes: {iris.target_names.tolist()}")
print(f"Samples/Class : {dict(df['species_name'].value_counts())}")
print("═" * 60)

print("\nFirst 5 rows:")
display(df.head())

print("\nStatistical Summary:")
display(df.describe())

## 3. Preprocessing and Data Verification

### WHY?
We must audit the dataset for null values or duplicate entries before evaluation. We separate features ($X$) and target labels ($y$) to prepare them for classification models.

### HOW?
We call `.isnull().sum()` to verify data quality and separate target variables from features.

In [ ]:
print("Checking for missing values:")
print(df[iris.feature_names].isnull().sum())

duplicates = df.duplicated().sum()
print(f"\nDuplicate rows found: {duplicates}")

# Separate Features (X) and Target (y)
X = iris.data
y = iris.target

print(f"\nFeature matrix X shape : {X.shape} (samples × features)")
print(f"Target vector y shape  : {y.shape} (samples)")
print(f"Class counts           : {dict(zip(*np.unique(y, return_counts=True)))}")

print("\nData is verified and ready for split tests.")

## 4. Lesson 5.2: Manual K-Fold Cross-Validation

### WHY?
Writing a K-Fold cross-validation loop manually using NumPy indexes is highly educational. It helps you understand what happens under the hood of Scikit-learn and prepares you for technical placement questions.

### HOW?
We permute indices randomly, divide them into $K=5$ equal chunks, train a `LogisticRegression` model on $K-1$ chunks, test on the remaining chunk, and repeat this $K$ times.

In [ ]:
k = 5
n_samples = len(X)

# Shuffle indices randomly using fixed numpy seed
np.random.seed(42)
shuffled_indices = np.random.permutation(n_samples)

# Split indices into k equal folds
folds = np.array_split(shuffled_indices, k)

print(f"Dataset split into k={k} folds successfully.")
for i, fold in enumerate(folds):
    print(f"  Fold {i+1}: size={len(fold)}, sample indices={fold[:5]}...")

manual_fold_accuracies = []

print("\n" + "─" * 60)
print(" Manual K-Fold Cross-Validation Loop")
print("─" * 60)

for fold_idx in range(k):
    # Select the current fold as validation data
    val_idx = folds[fold_idx]
    
    # Combine all other folds to form training data
    train_idx = np.concatenate([folds[j] for j in range(k) if j != fold_idx])
    
    # Slice features and target arrays
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Train a new model for each fold to avoid data leakage
    model = LogisticRegression(max_iter=200, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate accuracy
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    manual_fold_accuracies.append(acc)
    
    print(f"  Fold {fold_idx + 1}: Train size={len(X_train)}, Val size={len(X_val)}, Accuracy={acc:.4f}")

print("─" * 60)

## 5. Summary Statistics Calculation

### WHY?
Model performance is incomplete without reporting both the mean and the standard deviation. The mean shows model quality; the standard deviation shows stability.

### HOW?
We calculate the mean and standard deviation of the accuracy scores using NumPy, mapping them to the mathematical formulas from Lesson 5.2.

In [ ]:
mean_acc = np.mean(manual_fold_accuracies)
std_acc = np.std(manual_fold_accuracies)

print("═" * 50)
print(" Manual K-Fold Summary Results")
print("═" * 50)
for i, acc in enumerate(manual_fold_accuracies):
    bar = "█" * int(acc * 25)
    print(f"  Fold {i+1}: Accuracy = {acc:.4f} {bar}")
print("─" * 50)
print(f"  Mean Accuracy : {mean_acc:.4f} ({mean_acc*100:.2f}%)")
print(f"  Std Deviation : {std_acc:.4f} ({std_acc*100:.2f}%)")
print(f"  Reported Score: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")
print("═" * 50)

if std_acc < 0.03:
    print("\nStability check: STABLE (standard deviation is low).")
else:
    print("\nStability check: MODERATE/HIGH variance. Auditing recommended.")

## 6. Lesson 5.3 & 5.5: Scikit-learn Cross-Validation

### WHY?
Using Scikit-learn splitters (`KFold`, `StratifiedKFold`) and wrappers (`cross_val_score`) simplifies evaluations and makes your code production-ready.

### HOW?
We run cross-validation using Scikit-learn tools, comparing standard K-Fold against Stratified K-Fold.

In [ ]:
model = LogisticRegression(max_iter=200, random_state=42)

# ----------------------------------------------------------------
# Standard KFold Splitter
# ----------------------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
sklearn_kfold_scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')

print("═" * 50)
print(" Scikit-learn Standard KFold Scores")
print("═" * 50)
for i, val in enumerate(sklearn_kfold_scores):
    print(f"  Fold {i+1}: Accuracy = {val:.4f}")
print(f"  Summary: {sklearn_kfold_scores.mean():.4f} ± {sklearn_kfold_scores.std():.4f}")

# ----------------------------------------------------------------
# Stratified KFold Splitter (Preserves class proportions)
# ----------------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sklearn_stratified_scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print("\n" + "═" * 50)
print(" Scikit-learn Stratified KFold Scores")
print("═" * 50)
for i, val in enumerate(sklearn_stratified_scores):
    print(f"  Fold {i+1}: Accuracy = {val:.4f}")
print(f"  Summary: {sklearn_stratified_scores.mean():.4f} ± {sklearn_stratified_scores.std():.4f}")

## 7. Performance and Distribution Visualizations

### WHY?
Visualizing performance across folds helps identify anomalous drops, while boxplots show the distribution and spread of scores.

### HOW?
We plot a side-by-side comparison using Matplotlib: a bar chart of fold accuracies, and a boxplot showing the score distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fold_nums = np.arange(1, 6)
bar_width = 0.25

# Left: Bar Chart comparison
axes[0].bar(fold_nums - bar_width, manual_fold_accuracies, width=bar_width, label='Manual K-Fold', color='#2196F3')
axes[0].bar(fold_nums, sklearn_kfold_scores, width=bar_width, label='Sklearn K-Fold', color='#FF9800')
axes[0].bar(fold_nums + bar_width, sklearn_stratified_scores, width=bar_width, label='Stratified K-Fold', color='#4CAF50')

axes[0].set_title('Per-Fold Accuracy by Cross-Validation Method', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Fold Number', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_xticks(fold_nums)
axes[0].set_ylim(0.85, 1.05)
axes[0].legend(title='Methods')

# Right: Boxplot showing score distributions
methods_data = [manual_fold_accuracies, sklearn_kfold_scores, sklearn_stratified_scores]
methods_labels = ['Manual K-Fold', 'Sklearn K-Fold', 'Stratified K-Fold']
colors = ['#2196F3', '#FF9800', '#4CAF50']

bp = axes[1].boxplot(methods_data, patch_artist=True, labels=methods_labels,
                     medianprops=dict(color='white', linewidth=2))

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Overlay individual scores to show exact distribution points
for i, data in enumerate(methods_data):
    jitter = np.random.uniform(-0.06, 0.06, size=len(data))
    axes[1].scatter(np.full(len(data), i + 1) + jitter, data, color=colors[i], edgecolors='black', zorder=3, s=50)

axes[1].set_title('Score Distribution Across Folds (Spread)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Validation Strategy', fontsize=11)
axes[1].set_ylabel('Accuracy Range', fontsize=11)
axes[1].set_ylim(0.85, 1.05)

plt.suptitle('Cross-Validation Performance Comparison', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8. Lesson 5.6: Empirical Sweep Over $K$-Fold Values

### WHY?
Choosing the number of folds ($K$) involves a trade-off. A small $K$ is fast but has high bias (trained on less data); a large $K$ has low bias but high computational cost. We run an empirical sweep to see this trade-off in action.

### HOW?
We loop through $K = [3, 5, 10]$, track the accuracy and training time, and display a comparison table.

In [ ]:
k_values = [3, 5, 10]
sweep_results = []

print("Running cross-validation for K = 3, 5, 10 ...")

for k_val in k_values:
    # Standardized stratified splits
    cv_splitter = StratifiedKFold(n_splits=k_val, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv_splitter, scoring='accuracy')
    
    # Compute metrics
    train_size_per_fold = int((k_val - 1) / k_val * len(X))
    val_size_per_fold = int(len(X) / k_val)
    
    sweep_results.append({
        'K (Folds)': k_val,
        'Mean Accuracy (%)': round(scores.mean() * 100, 2),
        'Standard Dev (%)': round(scores.std() * 100, 2),
        'Train Size/Fold': train_size_per_fold,
        'Val Size/Fold': val_size_per_fold,
        'Total Models': k_val
    })

sweep_df = pd.DataFrame(sweep_results)

print("\n" + "═" * 75)
print("                       K-FOLD VALUE SWEEP COMPARISON")
print("═" * 75)
display(sweep_df.set_index('K (Folds)'))
print("═" * 75)

print("\n📌 Trade-off Analysis:")
print("  K=3  → Trains on 67% of data per fold. Fast, but higher bias (model has less data to learn from).")
print("  K=5  → Trains on 80% of data per fold. The industry-standard sweet spot.")
print("  K=10 → Trains on 90% of data per fold. Lower bias, but takes twice as long to compute.")

# Part 9: Placement & Interview Q&A

**Q1. What is Cross-Validation and why is it preferred over a single train-test split?**  
**Answer:** Cross-validation is a resampling technique that evaluates models by training and testing on multiple different subsets of the data. It is preferred over a single train-test split because it reduces validation variance, providing a more stable and generalizable estimate of model performance.

**Q2. When should you use Stratified K-Fold instead of standard K-Fold?**  
**Answer:** Stratified K-Fold should be used when dealing with imbalanced datasets. It ensures that target class proportions are preserved in each fold, preventing situations where some folds contain no examples of the minority class.

**Q3. Why is random shuffling dangerous for time-series validation?**  
**Answer:** Time-series data has temporal dependencies. Random shuffling would allow the model to train on future data points to predict past data points, causing data leakage and creating unrealistic performance scores that fail in production.

**Q4. Explain the trade-offs of Leave-One-Out Cross-Validation (LOOCV).**  
**Answer:** LOOCV maximizes the data available for training by using $n-1$ samples in each fold. However, it is computationally expensive since it requires training the model $n$ times, and the resulting validation scores can have high variance.

**Q5. Why should preprocessing steps like scaling be performed inside the cross-validation loop rather than before it?**  
**Answer:** Preprocessing before splitting leaks information from the validation fold (such as the mean and standard deviation) into the training folds. This data leakage inflates evaluation scores, making the model look more accurate than it actually is.

---

# Key Takeaways

- **Cross-validation provides a stable estimate of model performance.** Shuffling the data and averaging scores across multiple folds prevents you from relying on a "lucky" or "unlucky" random split.
- **Class distribution dictates your splitting strategy.** Always use Stratified K-Fold for classification tasks to keep target class proportions consistent across folds.
- **Respect time dependencies.** Shuffling breaks the sequential order of time-series data. Use Time Series Cross-Validation to evaluate models on chronological, forward-looking splits.
- **Beware of data leakage.** Preprocessing steps (like scaling) must be fit inside each cross-validation fold, not on the entire dataset beforehand. Use Scikit-learn Pipelines to automate this process and prevent leaks.
- **Report both mean and standard deviation.** A model with high average accuracy but high variance (standard deviation) is unstable. Always report the standard deviation to give a complete picture of model reliability.